# 01 — Làm sạch KHÔNG bỏ dòng trùng

Tái hiện đúng quy trình làm sạch của `notebooks/01_Data_Cleaning.ipynb`, **trừ bước
xoá dòng trùng lặp**.

**Cách ly:** notebook này chỉ **đọc** `../../data/raw/` và `../../data/cleaned/`
(để đối chứng). Nó chỉ **ghi** vào `experiment_nodedup/`. Các thư mục
`notebooks/`, `data/cleaned/`, `models/`, `reports/` của project gốc không bị động tới.

| Biến thể | Bỏ dòng trùng | Bỏ dòng mâu thuẫn |
|---|---|---|
| `V1_nodedup` | **không** | có |
| `V2_nodedup_nocontra` | **không** | **không** |
| `V0_official_recipe` | có | có (đối chứng) |

`V0` phải dựng lại trùng khít từng giá trị với bộ cleaned chính thức — nếu không,
mọi so sánh sau đó mất tính đối chứng.

In [1]:
import json, os
import numpy as np
import pandas as pd
from fastai.tabular.all import df_shrink

# notebook nằm ở experiment_nodedup/notebooks -> project gốc là thư mục cha của cha
BASE = os.path.abspath(os.path.join('..', '..'))
NEW = os.path.join(BASE, 'experiment_nodedup')
RAW = os.path.join(BASE, 'data', 'raw', 'CSV Files', 'Training and Testing Sets')
OUTD = os.path.join(NEW, 'data', 'cleaned_nodedup')
OUTR = os.path.join(NEW, 'reports')
for p in [OUTD, OUTR, os.path.join(NEW, 'models')]:
    os.makedirs(p, exist_ok=True)
print('project gốc :', BASE)
print('ghi vào     :', NEW)

C:\venvs\ids5g\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


project gốc : G:\.shortcut-targets-by-id\1C2Fd9KvthwrCJvr4mxV0l6w7K1lsUbNE\5G_IDS_Project
ghi vào     : G:\.shortcut-targets-by-id\1C2Fd9KvthwrCJvr4mxV0l6w7K1lsUbNE\5G_IDS_Project\experiment_nodedup


In [2]:
CONTAMINATING = ['ct_state_ttl', 'sttl', 'dttl', 'ct_dst_ltm', 'ct_srv_dst',
                 'ct_dst_src_ltm', 'ct_srv_ltm', 'ct_srv_src', 'ct_dst_sport_ltm']
USELESS = ['is_sm_ips_ports', 'ct_ftp_cmd', 'is_ftp_login']
TARGETS = ['label', 'attack_cat']


def clean(csv_path, drop_dups, drop_contradictions):
    """Giống hàm clean() của notebook 01; hai bước xoá dòng được bật/tắt bằng cờ."""
    df = pd.read_csv(csv_path, skipinitialspace=True, low_memory=False, encoding='utf-8')
    st = {'rows_raw': len(df), 'attack_rate_raw': float(df['label'].mean())}
    df.drop(columns=['id'], inplace=True)
    df = df_shrink(df, skip=[], obj2cat=True, int2uint=False)
    st['nan'] = int(df.isna().sum().sum())
    to_drop = CONTAMINATING + USELESS
    st['not_in_data'] = [c for c in to_drop if c not in df.columns]
    df.drop(columns=[c for c in to_drop if c in df.columns], inplace=True)

    feat = [c for c in df.columns if c not in TARGETS]
    st['duplicate_rows_present'] = int(df.duplicated().sum())
    if drop_dups:
        n = len(df); df = df.drop_duplicates(); st['duplicates_removed'] = n - len(df)
    else:
        st['duplicates_removed'] = 0

    n_lab = df.groupby(feat, observed=True, dropna=False)['label'].transform('nunique')
    st['contradictory_present'] = int((n_lab > 1).sum())
    if drop_contradictions:
        df = df[n_lab == 1]; st['contradictory_removed'] = st['contradictory_present']
    else:
        st['contradictory_removed'] = 0

    df = df.reset_index(drop=True)
    st['rows_final'], st['cols_final'] = df.shape
    st['attack_rate_final'] = float(df['label'].mean())
    return df, st

In [3]:
variants = {
    'V1_nodedup':          dict(drop_dups=False, drop_contradictions=True),
    'V2_nodedup_nocontra': dict(drop_dups=False, drop_contradictions=False),
    'V0_official_recipe':  dict(drop_dups=True,  drop_contradictions=True),
}
stats, frames = {}, {}
for vname, kw in variants.items():
    stats[vname] = {}
    for split in ['training', 'testing']:
        df, st = clean(os.path.join(RAW, f'UNSW_NB15_{split}-set.csv'), **kw)
        stats[vname][split] = st; frames[(vname, split)] = df
        print(f"[{vname}/{split}] {st['rows_raw']:,} -> {st['rows_final']:,} dòng | "
              f"Attack {st['attack_rate_raw']:.4f} -> {st['attack_rate_final']:.4f} | "
              f"trùng: có {st['duplicate_rows_present']:,} xoá {st['duplicates_removed']:,} | "
              f"mâu thuẫn: có {st['contradictory_present']:,} xoá {st['contradictory_removed']:,}")

[V1_nodedup/training] 175,341 -> 170,515 dòng | Attack 0.6806 -> 0.6807 | trùng: có 75,775 xoá 0 | mâu thuẫn: có 4,826 xoá 4,826


[V1_nodedup/testing] 82,332 -> 80,816 dòng | Attack 0.5506 -> 0.5560 | trùng: có 31,523 xoá 0 | mâu thuẫn: có 1,516 xoá 1,516


[V2_nodedup_nocontra/training] 175,341 -> 175,341 dòng | Attack 0.6806 -> 0.6806 | trùng: có 75,775 xoá 0 | mâu thuẫn: có 4,826 xoá 0


[V2_nodedup_nocontra/testing] 82,332 -> 82,332 dòng | Attack 0.5506 -> 0.5506 | trùng: có 31,523 xoá 0 | mâu thuẫn: có 1,516 xoá 0


[V0_official_recipe/training] 175,341 -> 98,934 dòng | Attack 0.6806 -> 0.4964 | trùng: có 75,775 xoá 75,775 | mâu thuẫn: có 632 xoá 632


[V0_official_recipe/testing] 82,332 -> 50,567 dòng | Attack 0.5506 -> 0.3665 | trùng: có 31,523 xoá 31,523 | mâu thuẫn: có 242 xoá 242


### Đối chứng bắt buộc

Biến thể `V0_official_recipe` phải tái lập **trùng khít** `data/cleaned`.

In [4]:
for split in ['training', 'testing']:
    official = pd.read_parquet(os.path.join(BASE, 'data', 'cleaned', f'UNSW_NB15_{split}-set.parquet'))
    mine = frames[('V0_official_recipe', split)][list(official.columns)].reset_index(drop=True)
    same = mine.equals(official.reset_index(drop=True))
    print(f'[{split}] V0 == data/cleaned parquet -> {same}')
    assert same, 'Hàm làm sạch không tái lập được bộ chính thức'
print('\nHàm làm sạch là bản sao trung thực. Mọi so sánh phía sau có đối chứng.')

[training] V0 == data/cleaned parquet -> True
[testing] V0 == data/cleaned parquet -> True

Hàm làm sạch là bản sao trung thực. Mọi so sánh phía sau có đối chứng.


## Bước làm sạch có tạo ra lệch tỷ lệ lớp không?

Báo cáo cũ khẳng định bước làm sạch **tự nó** tạo ra lệch 13 điểm giữa train và test.
Ở đây đo trực tiếp: so lệch **trước** và **sau** khi làm sạch.

In [5]:
gap = pd.DataFrame([{
    'biến thể': v,
    'train thô': stats[v]['training']['attack_rate_raw'],
    'test thô': stats[v]['testing']['attack_rate_raw'],
    'lệch thô': stats[v]['training']['attack_rate_raw'] - stats[v]['testing']['attack_rate_raw'],
    'train sạch': stats[v]['training']['attack_rate_final'],
    'test sạch': stats[v]['testing']['attack_rate_final'],
    'lệch sạch': stats[v]['training']['attack_rate_final'] - stats[v]['testing']['attack_rate_final'],
} for v in variants]).set_index('biến thể')
display(gap.round(4))

v0 = stats['V0_official_recipe']
print(f"Làm sạch chính thức kéo tỷ lệ Attack của train xuống "
      f"{v0['training']['attack_rate_raw'] - v0['training']['attack_rate_final']:.4f} điểm")
print(f"                               của test  xuống "
      f"{v0['testing']['attack_rate_raw'] - v0['testing']['attack_rate_final']:.4f} điểm")
print('=> Hai tập bị kéo xuống gần như BẰNG NHAU, nên khoảng lệch không đổi.')

,train thô,test thô,lệch thô,train sạch,test sạch,lệch sạch
biến thể,,,,,,
V1_nodedup,0.6806,0.5506,0.13,0.6807,0.5560,0.1247
V2_nodedup_nocontra,0.6806,0.5506,0.13,0.6806,0.5506,0.1300
V0_official_recipe,0.6806,0.5506,0.13,0.4964,0.3665,0.1299


Làm sạch chính thức kéo tỷ lệ Attack của train xuống 0.1842 điểm
                               của test  xuống 0.1841 điểm
=> Hai tập bị kéo xuống gần như BẰNG NHAU, nên khoảng lệch không đổi.


## Cái giá của việc giữ lại dòng trùng: rò rỉ test vào train

In [6]:
cross = {}
for vname in variants:
    tr, te = frames[(vname, 'training')], frames[(vname, 'testing')]
    key = [c for c in tr.columns if c not in TARGETS] + ['label']
    tr_keys = set(tr[key].astype(str).agg('|'.join, axis=1))
    hit = te[key].astype(str).agg('|'.join, axis=1).isin(tr_keys)
    cross[vname] = {'dòng test': len(te), 'trùng với train': int(hit.sum()), 'tỷ lệ': float(hit.mean())}
display(pd.DataFrame(cross).T.assign(**{'tỷ lệ': lambda d: d['tỷ lệ'].map('{:.2%}'.format)}))

,dòng test,trùng với train,tỷ lệ
V1_nodedup,80816.0,20960.0,25.94%
V2_nodedup_nocontra,82332.0,23536.0,28.59%
V0_official_recipe,50567.0,1993.0,3.94%


In [7]:
for vname in ['V1_nodedup', 'V2_nodedup_nocontra']:
    for split in ['training', 'testing']:
        p = os.path.join(OUTD, f'{vname}_UNSW_NB15_{split}-set.parquet')
        frames[(vname, split)].to_parquet(p)
        print('đã lưu', os.path.relpath(p, BASE))

with open(os.path.join(OUTR, 'cleaning_stats.json'), 'w', encoding='utf-8') as f:
    json.dump({'stats': stats, 'prevalence_gap': gap.round(6).to_dict('index'),
               'cross_set_duplicates': cross}, f, ensure_ascii=False, indent=2)
print('đã lưu reports/cleaning_stats.json')

đã lưu experiment_nodedup\data\cleaned_nodedup\V1_nodedup_UNSW_NB15_training-set.parquet
đã lưu experiment_nodedup\data\cleaned_nodedup\V1_nodedup_UNSW_NB15_testing-set.parquet


đã lưu experiment_nodedup\data\cleaned_nodedup\V2_nodedup_nocontra_UNSW_NB15_training-set.parquet
đã lưu experiment_nodedup\data\cleaned_nodedup\V2_nodedup_nocontra_UNSW_NB15_testing-set.parquet
đã lưu reports/cleaning_stats.json
